# 03. Оценка ответов в Ragas

Во [второй части](02_retrieval_and_answer.ipynb) мы сохранили ответы вместе с точным контекстом, который видела модель. Теперь их проверяет LLM-судья через [Ragas](https://docs.ragas.io/). Главный вопрос: если ответ плохой, где система ошиблась — при поиске (нужного факта нет в контексте) или при генерации (факт был, но модель его исказила или пропустила)?

```text
вопрос + ответ + контексты + эталон → метрика Ragas → число от 0 до 1
```

Индекс и генератор здесь не нужны, только судья и embeddings для Answer Relevancy. Каждая метрика делает несколько запросов к судье, поэтому считаем их не на всём наборе, а на 8 вопросах, подобранных так, чтобы покрыть разные типы: формула, числа из таблицы, расхождение в самой статье, подписи к рисункам, английские вопросы. Это 40 оценок.

In [1]:
import sys
from pathlib import Path

# Ноутбук можно открыть из этой папки или из корня репозитория.
ROOT = Path.cwd() if (Path.cwd() / "helpers").is_dir() else Path.cwd() / "hybrid_rag_notebooks"
sys.path.insert(0, str(ROOT))

from helpers.settings import load_env  # noqa: E402

load_env(ROOT)

In [2]:
import math

from helpers.corpus import load_questions, sha256
from helpers.evaluation import (
    CONTEXT_METRICS,
    METRIC_NAMES,
    REFERENCE_METRICS,
    is_auth_error,
    mean_scores,
    score_table,
    status_counts,
)
from helpers.report import print_example
from helpers.settings import judge_api_key, judge_embed_api_key, judge_settings
from helpers.storage import append_jsonl, check_config, latest_records, read_json
from tqdm.auto import tqdm

RUN_NAME = "attention_v1_hybrid_rerank"
EVAL_NAME = "ragas_v1"  # при смене судьи задайте новое имя
RUN_DIR = ROOT / "outputs" / "runs" / RUN_NAME
EVAL_DIR = RUN_DIR / "evaluations" / EVAL_NAME
QUESTIONS_PATH = ROOT / "data" / "questions.jsonl"
JUDGE = judge_settings()

EVAL_IDS = [
    "attn_02_scaling",  # формула attention и зачем делить на √d_k
    "attn_07_qkv",  # en: откуда берутся Q, K, V
    "attn_08_head_dimensions",  # числа из текста: h, d_k, d_v
    "attn_17_complexity",  # таблица 1, сложность слоёв
    "attn_29_translation_table",  # таблица 2, BLEU и стоимость обучения
    "attn_30_french_discrepancy",  # в статье два разных числа BLEU EN-FR
    "attn_31_dev_test",  # en: dev и test в разных таблицах
    "attn_36_attention_figures",  # рисунки 3–5, описания от VLM
]

questions = {q["id"]: q for q in load_questions(QUESTIONS_PATH)}
if read_json(RUN_DIR / "config.json")["questions_sha256"] != sha256(QUESTIONS_PATH):
    raise ValueError("questions.jsonl изменился после генерации ответов; перезапустите 02")
answers = latest_records(RUN_DIR / "answers.jsonl")
missing = [i for i in EVAL_IDS if answers.get(i, {}).get("status") != "ok"]
assert not missing, f"нет ответов на {missing}; выполните 02"
print(f"{RUN_NAME}: {len(answers)} ответов, оцениваем {len(EVAL_IDS)}. Судья: {JUDGE['model']}")

attention_v1_hybrid_rerank: 40 ответов, оцениваем 8. Судья: gpt-6.1-sol


## 1. Что получает судья

- `user_input` — вопрос;
- `response` — сохранённый ответ;
- `retrieved_contexts` — те самые блоки `[1]`, `[2]`, … из промпта генератора;
- `reference` — эталонный ответ, написанный вручную по статье. Генератор его не видел.

In [3]:
sample = answers[EVAL_IDS[2]]
print("Вопрос:", sample["question"])
print("\nОтвет:", sample["response"])
print("\nЭталон:", questions[sample["id"]]["reference"])
print(
    f"\nКонтекстов: {len(sample['retrieved_contexts'])}, первый:\n{sample['retrieved_contexts'][0][:600]}…"
)

Вопрос: Сколько голов внимания используется в базовой модели и как связаны d_model, d_k, d_v и число голов h?

Ответ: В базовой модели используется **h = 8 голов внимания**. При этом \(d_{\text{model}} = 512\), а размерности ключей и значений каждой головы составляют \(d_k = d_v = 64\), то есть \(d_k = d_v = d_{\text{model}}/h\) [1][3]. Таким образом, \(512/8=64\).

Эталон: В базовой модели h = 8 голов при d_model = 512. Для каждой головы d_k = d_v = d_model / h = 64. Головы используют разные обучаемые линейные проекции, а их результаты объединяются и снова проецируются.

Контекстов: 5, первый:
[1] (Attention is all you need.pdf, p.4)
3.2.2 Multi-Head Attention
Instead of performing a single attention function with dmodel-dimensional keys, values and queries, we found it beneficial to linearly project the queries, keys and values h times with different, learned linear projections to dk, dk and dv dimensions, respectively. On each of these projected versions of queries, keys and values 

## 2. Метрики

| метрика | что проверяет | нужно |
|---|---|---|
| Faithfulness | каждое утверждение ответа подтверждается контекстом | ответ, контекст |
| Answer Relevancy | ответ по существу вопроса: судья придумывает вопросы к ответу, embeddings сравнивают их с исходным | вопрос, ответ |
| Context Precision | полезные контексты стоят выше бесполезных | контекст, эталон |
| Context Recall | всё, что есть в эталоне, можно найти в контексте | контекст, эталон |
| Factual Correctness | совпадение фактов ответа и эталона (F1 по утверждениям) | ответ, эталон |

Context Precision и Context Recall оценивают поиск, Faithfulness — генерацию, Factual Correctness — результат целиком. Так их и разбираем: низкий Context Recall означает, что ответа не было в контексте; высокий Recall при низком Factual Correctness — что модель не воспользовалась найденным.

Судья задаётся в `.env`. Если `JUDGE_*` пустые, судьёй работает та же LLM, что генерировала ответы, и тогда она может быть снисходительна к своим же ответам. В этом прогоне судья — отдельная модель (её имя напечатано выше).

In [4]:
from openai import AsyncOpenAI
from ragas.embeddings import OpenAIEmbeddings
from ragas.llms import llm_factory
from ragas.metrics.collections import (
    AnswerRelevancy,
    ContextPrecisionWithReference,
    ContextRecall,
    FactualCorrectness,
    Faithfulness,
)

judge_client = AsyncOpenAI(
    base_url=JUDGE["base_url"], api_key=judge_api_key(), timeout=JUDGE["timeout"], max_retries=2
)
embed_client = AsyncOpenAI(
    base_url=JUDGE["embedding_url"],
    api_key=judge_embed_api_key(),
    timeout=JUDGE["timeout"],
    max_retries=2,
)
llm = llm_factory(JUDGE["model"], client=judge_client, temperature=JUDGE["temperature"])
embeddings = OpenAIEmbeddings(client=embed_client, model=JUDGE["embedding_model"])

metrics = {
    "faithfulness": Faithfulness(llm=llm),
    "answer_relevancy": AnswerRelevancy(
        llm=llm, embeddings=embeddings, strictness=JUDGE["strictness"]
    ),
    "context_precision_with_reference": ContextPrecisionWithReference(llm=llm),
    "context_recall": ContextRecall(llm=llm),
    "factual_correctness": FactualCorrectness(llm=llm, mode="f1"),
}

## 3. Одна метрика вручную

Вызовем Faithfulness на одном ответе. Внутри это два шага: судья разбивает ответ на отдельные утверждения, затем для каждого решает, следует ли оно из контекста. Оценка — доля подтверждённых утверждений.

In [5]:
original = await metrics["faithfulness"].ascore(
    user_input=sample["question"],
    response=sample["response"],
    retrieved_contexts=sample["retrieved_contexts"],
)
print("Faithfulness:", round(original.value, 3))

Faithfulness: 1.0


Проверим, что метрика вообще реагирует на ошибку. Допишем к тому же ответу ложное утверждение о статье и посчитаем заново. Если оценка не упадёт, метрике в таком виде доверять нельзя.

In [6]:
broken = (
    sample["response"]
    + " Кроме того, в статье указано, что базовая модель обучалась на 64 GPU A100."
)
corrupted = await metrics["faithfulness"].ascore(
    user_input=sample["question"],
    response=broken,
    retrieved_contexts=sample["retrieved_contexts"],
)
print(f"Исходный ответ: {original.value:.3f}, с ложным утверждением: {corrupted.value:.3f}")

Исходный ответ: 1.000, с ложным утверждением: 0.833


## 4. Все метрики на выбранных вопросах

`score_one` знает, какие поля нужны каждой метрике. Если метрика неприменима (нет эталона, пустой контекст), ставим `skipped`. Если упал запрос — `error`, а не ноль: иначе сбои API тянули бы среднее вниз. Результат каждой пары «вопрос × метрика» сразу пишется в `scores.jsonl`, поэтому повторный запуск досчитывает только недостающее.

Пары считаем параллельно, по `CONCURRENCY` штук: последовательно 40 оценок шли бы около получаса.

In [7]:
import asyncio


async def score_one(row, name):
    reference = questions[row["id"]].get("reference")
    contexts = row.get("retrieved_contexts") or []
    if name in REFERENCE_METRICS and not reference:
        return {"status": "skipped", "reason": "нет эталона"}
    if name in CONTEXT_METRICS and not contexts:
        return {"status": "skipped", "reason": "пустой контекст"}
    inputs = {
        "faithfulness": dict(
            user_input=row["question"], response=row["response"], retrieved_contexts=contexts
        ),
        "answer_relevancy": dict(user_input=row["question"], response=row["response"]),
        "context_precision_with_reference": dict(
            user_input=row["question"], reference=reference, retrieved_contexts=contexts
        ),
        "context_recall": dict(
            user_input=row["question"], reference=reference, retrieved_contexts=contexts
        ),
        "factual_correctness": dict(response=row["response"], reference=reference),
    }[name]
    try:
        value = float((await metrics[name].ascore(**inputs)).value)
    except Exception as exc:
        if is_auth_error(exc):
            raise RuntimeError("судья ответил 401/403, проверьте JUDGE_API_KEY") from exc
        return {"status": "error", "error": f"{type(exc).__name__}: {exc}"[:300]}
    if not math.isfinite(value):
        return {"status": "undefined", "reason": "метрика вернула NaN"}
    return {"status": "ok", "value": value}


CONCURRENCY = 6
check_config(
    EVAL_DIR / "judge.json", {"run": RUN_NAME, "judge": JUDGE, "metrics": list(METRIC_NAMES)}
)
scores_path = EVAL_DIR / "scores.jsonl"
done = latest_records(scores_path, key=("id", "metric"))
work = [
    (i, m)
    for i in EVAL_IDS
    for m in METRIC_NAMES
    if done.get((i, m), {}).get("status") not in ("ok", "skipped")
]
print(f"Пар всего: {len(EVAL_IDS) * len(METRIC_NAMES)}, осталось посчитать: {len(work)}")

limit = asyncio.Semaphore(CONCURRENCY)


async def run_pair(sample_id, name):
    async with limit:
        result = await score_one(answers[sample_id], name)
    append_jsonl(scores_path, {"id": sample_id, "metric": name, **result})
    return result


for future in tqdm(asyncio.as_completed([run_pair(i, m) for i, m in work]), total=len(work)):
    result = await future
    if result["status"] == "error":
        tqdm.write(result["error"])

Пар всего: 40, осталось посчитать: 0


## 5. Результаты

In [8]:
scores = latest_records(scores_path, key=("id", "metric"))
table = score_table([questions[i] for i in EVAL_IDS], scores, run=RUN_NAME)
table.to_csv(EVAL_DIR / "details.csv", index=False)
status_counts(table)

status,ok
metric,
faithfulness,8
answer_relevancy,8
context_precision_with_reference,8
context_recall,8
factual_correctness,8


In [9]:
mean_scores(table)

metric,faith,relevancy,ctx_precision,ctx_recall,factual
run,,,,,
attention_v1_hybrid_rerank,1.0,0.826,0.89,1.0,0.551


Средние по восьми вопросам мало что говорят сами по себе. Полезнее таблица по вопросам: в ней видно, какой этап подвёл в каждом случае.

In [10]:
by_question = table.pivot(index="id", columns="metric", values="value")[list(METRIC_NAMES)]
by_question.columns = ["faith", "relevancy", "ctx_precision", "ctx_recall", "factual"]
by_question.astype(float).round(2).sort_values("factual")

,faith,relevancy,ctx_precision,ctx_recall,factual
id,,,,,
attn_29_translation_table,1.0,0.89,0.58,1.0,0.00
attn_30_french_discrepancy,1.0,0.81,1.00,1.0,0.33
attn_31_dev_test,1.0,0.69,1.00,1.0,0.33
attn_36_attention_figures,1.0,0.73,1.00,1.0,0.33
attn_07_qkv,1.0,0.73,0.95,1.0,0.67
attn_08_head_dimensions,1.0,0.87,0.76,1.0,0.75
attn_02_scaling,1.0,0.94,0.83,1.0,1.00
attn_17_complexity,1.0,0.95,1.00,1.0,1.00


## 6. Разбор худшего ответа

Берём вопрос с самой низкой Factual Correctness и читаем всё рядом: ответ, эталон и контекст. Порядок разбора такой:

1. Есть ли нужный факт в контексте? Если нет — проблема в поиске или в разборе PDF, возвращаемся в 01/02.
2. Факт есть, но ответ его исказил или пропустил — проблема в генерации или промпте.
3. По смыслу ответ верный, а метрика низкая — значит, ошибся судья или эталон слишком узкий.

In [11]:
worst = by_question["factual"].astype(float).idxmin()
print_example(worst, list(questions.values()), answers, table)

attn_29_translation_table: По таблице 2, какие BLEU получили Transformer base и Transformer big на English-to-German и English-to-French?
Где искать в PDF: стр. 8
                          metric status    value
                    faithfulness     ok 1.000000
                answer_relevancy     ok 0.894489
context_precision_with_reference     ok 0.583333
                  context_recall     ok 1.000000
             factual_correctness     ok 0.000000

Ответ:
| Модель | English-to-German | English-to-French |
|---|---:|---:|
| Transformer base | 27,3 BLEU | 38,1 BLEU |
| Transformer big | 28,4 BLEU | 41,8 BLEU |

Значения приведены в таблице 2 для теста newstest2014. [2]

Эталон:
В таблице 2 Transformer base имеет 27.3 BLEU для EN-DE и 38.1 для EN-FR. Transformer big имеет 28.4 BLEU для EN-DE и 41.8 для EN-FR. Ответ относится именно к значениям таблицы 2.

Контекст (13586 символов):
[1] (Attention is all you need.pdf, p.8)
6.1 Machine Translation
On the WMT 2014 English-to-German tran

### Почему верный ответ получил 0

Числа в ответе на attn_29 совпадают с эталоном. Factual Correctness работает в два шага: судья разбивает ответ на утверждения, а затем проверяет каждое **по эталону**, а не по статье. Посмотрим на оба шага. В публичном API Ragas их нет, поэтому вызываем внутренние методы метрики.

In [12]:
worst_answer = answers[worst]["response"]
worst_reference = questions[worst]["reference"]
fc = metrics["factual_correctness"]
claims = await fc._decompose_claims(worst_answer)
verdicts = await fc._verify_claims(claims, worst_reference)
for item in verdicts.statements:
    print(f"[{'да' if item.verdict else 'нет'}] {item.statement}\n      судья: {item.reason}\n")

[нет] На тесте newstest2014 модель Transformer base достигает 27,3 BLEU для перевода с английского на немецкий и 38,1 BLEU для перевода с английского на французский.
      судья: Значения BLEU для Transformer base совпадают с контекстом, но в контексте не указано, что они получены на тесте newstest2014.

[нет] На тесте newstest2014 модель Transformer big достигает 28,4 BLEU для перевода с английского на немецкий и 41,8 BLEU для перевода с английского на французский.
      судья: Значения BLEU для Transformer big совпадают с контекстом, но в контексте не указано, что они получены на тесте newstest2014.

[нет] Результаты моделей Transformer base и Transformer big на тесте newstest2014 приведены в таблице 2.
      судья: Контекст относит результаты обеих моделей к таблице 2, но не связывает их с тестом newstest2014.


Все утверждения отклонены по одной причине: в ответе сказано «newstest2014», а в эталоне этого слова нет. Деталь при этом верная, она есть в подписи к таблице 2. Метрика сверяет ответ с эталоном, а не со статьёй, поэтому любой верный факт сверх эталона считается неподтверждённым. Судья приклеил «newstest2014» к каждому утверждению с числами, и они провалились целиком, вместе с правильными значениями BLEU.

Попробуем более мелкое разбиение на утверждения (`atomicity="high"`): может быть, тогда числа и набор данных окажутся в разных утверждениях.

In [13]:
fine = FactualCorrectness(llm=llm, mode="f1", atomicity="high")
result = await fine.ascore(response=worst_answer, reference=worst_reference)
print(
    f"atomicity=low (по умолчанию): {scores[(worst, 'factual_correctness')]['value']:.2f}, "
    f"atomicity=high: {result.value:.2f}"
)

atomicity=low (по умолчанию): 0.00, atomicity=high: 0.00


Остальные низкие значения Factual Correctness (0.33) — у ответов, которые по существу верны. F1 складывается из двух половин: precision (утверждения ответа подтверждаются эталоном) и recall (утверждения эталона есть в ответе). Посчитаем их отдельно для вопроса о рисунках.

In [14]:
figures = answers["attn_36_attention_figures"]
for mode in ("precision", "recall"):
    part = await FactualCorrectness(llm=llm, mode=mode).ascore(
        response=figures["response"], reference=questions["attn_36_attention_figures"]["reference"]
    )
    print(f"{mode}: {part.value:.2f}")

precision: 0.67
recall: 0.50


In [15]:
await judge_client.close()
await embed_client.close()

## Что получилось

На восьми вопросах:

- **Faithfulness 1.0 и Context Recall 1.0 у всех ответов.** Модель не выходит за пределы контекста, а всё, что есть в эталоне, в контекст попадает. Это согласуется со второй частью: с reranker нужные страницы были в контексте для всех вопросов.
- **Метрика реагирует на ошибку**: ложное утверждение про A100, дописанное к ответу, снизило Faithfulness с 1.0 до 0.83, то есть одно утверждение из шести не подтвердилось.
- **Factual Correctness (в среднем 0.55) — единственная низкая метрика, но она говорит не о качестве ответов.** Прочитав ответы с низкой оценкой, мы не нашли в них фактических ошибок. Проблема в устройстве метрики: она проверяет ответ по эталону, а не по статье. Ответ на attn_29 получил 0 из-за верного упоминания newstest2014, которого нет в эталоне. Более мелкое разбиение (`atomicity="high"`) не помогло, оценка осталась 0.
- **Судья нестабилен.** Precision и recall для вопроса о рисунках в этом прогоне 0.67 и 0.50, а в предыдущем запуске тех же ячеек были 0.33 и 0.60. Пересказ ответа attn_29 обычным текстом в одном запуске получил 0.8, в другом 0. Значения Factual Correctness на отдельном вопросе нельзя сравнивать между запусками с точностью лучше нескольких десятых.
- **Context Precision (0.89)** ниже всего у attn_29 (0.58): первым в контексте стоит текст раздела 6.1, где для EN-FR указано 41.0, а сама таблица 2 идёт вторым блоком.

Что из этого следует для работы с Ragas:
1. Средние значения без чтения ответов вводят в заблуждение: худший по метрике ответ оказался полностью верным.
2. Faithfulness и Context Recall здесь надёжны и хорошо разделяют ошибки поиска и генерации. Factual Correctness с коротким эталоном систематически наказывает ответы, которые подробнее эталона.
3. Если нужна Factual Correctness, эталон должен быть полным: включать всё, что верно и может прозвучать в ответе (набор данных, условия, подписи). Либо стоит смотреть precision и recall по отдельности и усреднять по нескольким запускам судьи.

Ограничения: одна статья, восемь вопросов, одна модель-судья, по одному запуску на пару. Различия меньше 0.1 здесь неотличимы от шума судьи.